# Clase 14B — RAG completo con dataset real de planes de gobierno
**Autor:** Josef Rodríguez

En esta clase construiremos una introducción seria y completa a **Retrieval Augmented Generation (RAG)** usando el dataset real del curso:

`https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv`

# 1. ¿Qué problema resuelve RAG?

Los LLMs son poderosos, pero tienen tres limitaciones estructurales:

1. **Knowledge cutoff**: no necesariamente conocen documentos recientes.
2. **Hallucinations**: pueden responder con seguridad aunque la respuesta no esté sustentada.
3. **No acceden por defecto a tu base documental privada**.

RAG resuelve esto combinando:
- **recuperación de información**,
- **representaciones semánticas**,
- **generación con LLMs**.

## 1.1 Diagrama de alto nivel

```text
Pregunta del usuario
        ↓
Embedding de la pregunta
        ↓
Búsqueda en base vectorial
        ↓
Top-K fragmentos relevantes
        ↓
Construcción del prompt
        ↓
LLM
        ↓
Respuesta final
```

## Idea clave
El LLM no responde solo con "memoria interna".  
Primero se le entrega **contexto relevante recuperado**.

# 2. Fundamento conceptual: RAG no es un modelo, es una arquitectura

Esto es muy importante para tus alumnos:

- Un embedding **no es** RAG.
- FAISS **no es** RAG.
- El LLM **no es** RAG.
- RAG es la combinación orquestada de varios componentes.

## Componentes típicos
1. Corpus documental
2. Chunking
3. Embeddings
4. Vector database
5. Retriever
6. Prompt builder
7. LLM
8. Post-procesamiento / evaluación

# 3. Cargar el dataset real

Trabajaremos con los textos de planes de gobierno y los dividiremos en fragmentos para que puedan ser recuperados con mayor precisión.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/josefrodrim/ML-course/main/data/planes_gobierno_texto.csv"
df = pd.read_csv(url)
df = df.rename(columns={df.columns[0]: "presidente", df.columns[1]: "texto_plan"})
df["texto_plan"] = df["texto_plan"].fillna("").astype(str)

df.head()

# 4. ¿Por qué hacer chunking?

Un documento completo suele ser demasiado largo para:
- indexarlo de manera fina,
- recuperar evidencia específica,
- alimentar eficientemente a un LLM.

Por eso dividimos el documento en **chunks**.

## Definición
Chunking = proceso de dividir un documento largo en fragmentos manejables.

## Diseño
Algunas decisiones importantes:
- tamaño del chunk
- overlap
- separación por párrafos, frases o ventanas de palabras

## 4.1 Estrategia simple de chunking por palabras

Si usamos tamaño \(L\) y overlap \(o\), el chunk \(i\) puede pensarse como una ventana:

\[
chunk_i = tokens[s_i : s_i + L]
\]

donde el inicio del siguiente chunk puede avanzar en:

\[
step = L - o
\]

Esto reduce el riesgo de cortar ideas importantes.

In [ ]:
def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []
    step = max(chunk_size - overlap, 1)
    for i in range(0, len(words), step):
        chunk = words[i:i+chunk_size]
        if len(chunk) == 0:
            continue
        chunks.append(" ".join(chunk))
        if i + chunk_size >= len(words):
            break
    return chunks

# Ejemplo con el primer documento
ej_chunks = chunk_text(df["texto_plan"].iloc[0], chunk_size=120, overlap=30)
len(ej_chunks), ej_chunks[0][:500]

# 5. Crear una base de chunks

Cada chunk será una unidad recuperable dentro del sistema RAG.

In [ ]:
rows = []
for _, row in df.iterrows():
    candidato = row["presidente"]
    texto = row["texto_plan"]
    chunks = chunk_text(texto, chunk_size=180, overlap=40)
    for j, ch in enumerate(chunks):
        rows.append({
            "presidente": candidato,
            "chunk_id": j,
            "chunk_text": ch
        })

chunks_df = pd.DataFrame(rows)
chunks_df.head()

In [ ]:
chunks_df.shape

# 6. Embeddings de chunks

Cada fragmento se convierte en un vector denso.  
Esta será la base de nuestra recuperación semántica.

In [ ]:
# Si estás en un entorno limpio, descomenta:
# !pip install -q sentence-transformers faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embed_model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True
)

chunk_embeddings = np.array(chunk_embeddings)
chunk_embeddings.shape

# 7. Base vectorial con FAISS

FAISS es una librería de Meta muy usada para búsqueda de vecinos cercanos en espacios vectoriales.

## Idea matemática
Dado un embedding de consulta \(q\), queremos encontrar los \(k\) vectores \(x_i\) más cercanos según una métrica como:
- similitud coseno,
- distancia L2,
- inner product.

In [ ]:
import faiss

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings.astype("float32"))

index.ntotal

# 8. Retriever: recuperar los chunks más relevantes

Nuestro retriever recibirá una consulta en lenguaje natural y devolverá los chunks más útiles como evidencia.

In [ ]:
def retrieve_chunks(query, top_k=5):
    q_emb = embed_model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    out = chunks_df.iloc[indices[0]].copy()
    out["distance_l2"] = distances[0]
    return out[["presidente", "chunk_id", "chunk_text", "distance_l2"]]

retrieve_chunks("propuestas sobre educación pública y calidad educativa", top_k=5)

In [ ]:
retrieve_chunks("hospitales, salud pública, atención primaria y telemedicina", top_k=5)

In [ ]:
retrieve_chunks("minería, exportaciones, economía y empleo", top_k=5)

# 9. Prompt construction

Una vez que recuperamos evidencia, construimos un prompt robusto.

## Plantilla recomendada

```text
Usa únicamente el contexto proporcionado para responder.
Si la respuesta no aparece en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
[chunk 1]
[chunk 2]
[chunk 3]

Pregunta:
[pregunta del usuario]
```

## Principio de diseño
Un buen RAG no solo recupera bien; también **instruye bien al LLM**.

In [ ]:
def build_prompt(query, retrieved_df):
    context = "\n\n".join(
        [f"[{r.presidente} | chunk {r.chunk_id}] {r.chunk_text}" for r in retrieved_df.itertuples()]
    )
    prompt = f"""Usa únicamente el contexto proporcionado para responder.
Si la respuesta no aparece en el contexto, responde: "No lo sé con la evidencia recuperada".

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""
    return prompt

prompt_demo = build_prompt(
    "¿Qué candidatos mencionan educación pública?",
    retrieve_chunks("educación pública", top_k=3)
)

print(prompt_demo[:2000])

# 10. Evaluación conceptual del retriever

Una masterclass no termina cuando el sistema "parece funcionar".  
También debemos enseñar cómo evaluarlo.

## Métricas comunes en retrieval
- Precision@k
- Recall@k
- MRR (Mean Reciprocal Rank)
- nDCG

## Ejemplo conceptual: Precision@k

Si de los \(k\) documentos recuperados, \(r\) son relevantes:

\[
Precision@k = \frac{{r}}{{k}}
\]

En entornos reales, la calidad del RAG depende muchísimo del retriever.

# 11. Limitaciones y decisiones de diseño

## Preguntas importantes
1. ¿Conviene chunking por palabras o por párrafos?
2. ¿Qué modelo de embedding conviene?
3. ¿Cuántos chunks recuperar?
4. ¿Conviene reranking adicional?
5. ¿Cómo evitar contexto redundante?

## Mensaje pedagógico
RAG no es solo "usar FAISS".  
RAG es una disciplina de diseño de sistemas.

# 12. Conclusiones

En esta clase aprendimos que:

1. RAG combina retrieval + prompting + generación.
2. Chunking es una decisión crítica.
3. Los embeddings permiten indexar significado, no solo palabras.
4. FAISS permite buscar rápidamente en grandes colecciones vectoriales.
5. El dataset real de planes de gobierno se presta muy bien para un sistema de preguntas y respuestas.

## Frase de cierre
**Un LLM útil para negocio no solo genera: primero busca evidencia.**